# E-Commerce Sales Analytics

**Tools:** Python, Pandas, SQLite SQL, Power BI

This notebook performs end-to-end exploratory and business analysis on 5,000 e-commerce sales transactions.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Find the raw CSV whether the notebook is launched from the repo root or python/ folder
candidates = [
    Path("data/raw/Ecommerce_Sales_Data_2024_2025.csv"),
    Path("../data/raw/Ecommerce_Sales_Data_2024_2025.csv")
]

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find the raw CSV. Run the notebook from the project folder.")

df = pd.read_csv(csv_path)
df.head()

## 1. Understand the dataset

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.info()

## 2. Clean and enrich the data

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Year-Month"] = df["Order Date"].dt.to_period("M").astype(str)
df["Profit Margin"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    np.nan
)

df.head()

## 3. KPI summary

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer Name"].nunique()
average_order_value = total_sales / total_orders
profit_margin = total_profit / total_sales * 100

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Average Order Value: ${average_order_value:,.2f}")
print(f"Profit Margin: {profit_margin:.2f}%")

## 4. Category performance

In [ ]:
category_performance = (
    df.groupby("Category")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
      .sort_values("Sales", ascending=False)
)
category_performance["Profit Margin %"] = (
    category_performance["Profit"] / category_performance["Sales"] * 100
)
category_performance.round(2)

## 5. Monthly sales trend

In [ ]:
monthly_sales = (
    df.groupby("Year-Month")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
)
monthly_sales

## 6. Top products

In [ ]:
top_products = (
    df.groupby("Product Name")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Quantity=("Quantity", "sum"))
      .sort_values("Sales", ascending=False)
      .head(10)
)
top_products

## 7. Region performance

In [ ]:
region_performance = (
    df.groupby("Region")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
      .sort_values("Sales", ascending=False)
)
region_performance

## 8. Payment method performance

In [ ]:
payment_performance = (
    df.groupby("Payment Mode")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
      .sort_values("Sales", ascending=False)
)
payment_performance

## 9. Top customers

In [ ]:
top_customers = (
    df.groupby("Customer Name")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
      .sort_values("Sales", ascending=False)
      .head(10)
)
top_customers

## 10. Top cities

In [ ]:
top_cities = (
    df.groupby("City")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
      .sort_values("Sales", ascending=False)
      .head(10)
)
top_cities

## 11. Discount analysis

In [ ]:
discount_analysis = (
    df.groupby("Discount")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
)
discount_analysis["Profit Margin %"] = (
    discount_analysis["Profit"] / discount_analysis["Sales"] * 100
)
discount_analysis.round(2)

## 12. Bottom products by profit

In [ ]:
bottom_products = (
    df.groupby("Product Name")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
      .sort_values("Profit", ascending=True)
      .head(10)
)
bottom_products

## 13. Yearly performance

In [ ]:
yearly_performance = (
    df.groupby("Year")
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Orders=("Order ID", "nunique"))
)
yearly_performance

# Important: 2023 and 2025 are partial years in this dataset.

## 14. Visualizations

In [ ]:
monthly_plot = (
    df.groupby("Year-Month")["Sales"].sum()
)
monthly_plot.plot(kind="line", figsize=(12, 5), marker="o", title="Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Sales")
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
(
    df.groupby("Category")["Sales"]
      .sum()
      .sort_values()
      .plot(kind="barh", figsize=(10, 5), title="Sales by Category")
)
plt.xlabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
(
    df.groupby("Region")["Sales"]
      .sum()
      .sort_values()
      .plot(kind="barh", figsize=(8, 5), title="Sales by Region")
)
plt.xlabel("Sales")
plt.tight_layout()
plt.show()

## 15. Load the cleaned dataset into SQLite

In [ ]:
conn = sqlite3.connect("ecommerce_project.db")

sql_df = df.copy()
sql_df["Order Date"] = sql_df["Order Date"].dt.strftime("%Y-%m-%d")
sql_df.to_sql("sales", conn, if_exists="replace", index=False)

pd.read_sql_query("SELECT * FROM sales LIMIT 5;", conn)

## 16. Example SQL analysis

In [ ]:
query = '''
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM sales
GROUP BY Category
ORDER BY Total_Sales DESC;
'''

pd.read_sql_query(query, conn)

## 17. Export cleaned data

In [ ]:
output_candidates = [
    Path("data/cleaned/ecommerce_sales_cleaned.csv"),
    Path("../data/cleaned/ecommerce_sales_cleaned.csv")
]
output_path = output_candidates[0] if output_candidates[0].parent.exists() else output_candidates[1]
df.to_csv(output_path, index=False)
print("Saved:", output_path)

## Final project takeaway

The analysis combines:
- data validation and cleaning
- KPI reporting
- category, product, region, customer, payment, and discount analysis
- time-series analysis
- SQL database analysis
- Power BI-ready data preparation

See `docs/Business_Insights_Resume_Interview.md` for findings and interview talking points.